## Modeling and Solving the Truss Problem
We are going to solve a simple 3-bar truss forming a right triangle.

**The Setup:**

* **Joint A** is at origin `(0, 0)`. It is secured to the wall with a **pin support**, meaning it provides reaction forces in both X and Y directions ($A_x$ and $A_y$).
* **Joint B** is at `(3, 0)`. It rests on a **roller support**, meaning it can slide in X, but provides a vertical reaction force ($B_y$).
* **Joint C** is at `(0, 4)`. It is a free joint experiencing an external load. Let's assume a force of $100$ N pushing right (positive X) and $50$ N pushing down (negative Y).

**The Members:**

* Member **AB** (Length = 3)
* Member **AC** (Length = 4)
* Member **BC** (Length = 5, forming a 3-4-5 right triangle)

Let's write the equilibrium equations ($\sum F_x = 0$ and $\sum F_y = 0$) for every joint, assuming that internal forces pull *away* from the joint (Tension is positive, compression is negative).


#### 1. Create a fresh solver


In [ ]:
truss_solver = Solver()

#### 2. Declare Variables

Six unknowns: one internal force per member, and one per reaction the supports can
provide. Every one of them is a real number, and we do not know the sign of any of them
in advance — that is part of what we are asking the solver for.


In [ ]:
# Internal forces in the members
T_AB = Real('T_AB')
T_AC = Real('T_AC')
T_BC = Real('T_BC')

# Reaction forces at the supports
A_x = Real('A_x')
A_y = Real('A_y')
B_y = Real('B_y')

#### 3. Add Constraints (Method of Joints)

Two equations per joint, six in total, one for each unknown. We write them down joint by
joint exactly as we would on paper, and we do not rearrange any of them.

In [ ]:
# --- JOINT A (0, 0) ---
# Forces acting on A: Reaction A, T_AB (horizontal), T_AC (vertical)
truss_solver.add(A_x + T_AB == 0)      # Sum of F_x = 0
truss_solver.add(A_y + T_AC == 0)      # Sum of F_y = 0

# --- JOINT B (3, 0) ---
# Forces acting on B: Reaction B_y, T_AB (pulling left), T_BC (pulling up-left towards C)
# The vector from B to C is (-3, 4). The unit vector components are X: -3/5, Y: 4/5.
truss_solver.add(-T_AB + T_BC * (-3.0/5.0) == 0)   # Sum of F_x = 0
truss_solver.add(B_y + T_BC * (4.0/5.0) == 0)      # Sum of F_y = 0

# --- JOINT C (0, 4) ---
# Forces acting on C: External load (100N right, 50N down), T_AC (pulling down), T_BC (pulling down-right towards B)
# The vector from C to B is (3, -4). The unit vector components are X: 3/5, Y: -4/5.
truss_solver.add(100 + T_BC * (3.0/5.0) == 0)      # Sum of F_x = 0
truss_solver.add(-50 - T_AC + T_BC * (-4.0/5.0) == 0)  # Sum of F_y = 0

# Let's see the six equations we have just written
showSolver(truss_solver)

#### 4. Solve the Truss


In [ ]:
# Check if solution exists
print( truss_solver.check() )

In [ ]:
# View solution
solution = truss_solver.model()
print( solution )

"sat" means all six equations can hold at once, and since there are six equations and
six unknowns, the assignment the solver returns is the only one. That assignment is the
solved truss.

Notice what the signs are telling us. $T_{AB}$ and $T_{AC}$ come out positive, so members
AB and AC are being pulled — they are in tension. $T_{BC}$ comes out negative, so the
diagonal is being pushed: it is in compression, which is what we would expect of the
member bracing the structure against a load pushing it sideways.

To better understand what is happening structurally, we can plot the truss. We will extract the solved forces and map them to our nodes.

* **Blue lines** will represent members in **tension** (pulling).
* **Red lines** will represent members in **compression** (pushing).
* Line thickness will represent the magnitude of the force.

In [ ]:
f_AB = float(solution[T_AB].as_fraction())
f_AC = float(solution[T_AC].as_fraction())
f_BC = float(solution[T_BC].as_fraction())
visualize_truss_solution( f_AB, f_AC, f_BC )

#### 5. Taking the roller away

The roller at B is doing something, but it is not obvious from the drawing how much.
Let's find out by removing it.

Everything else stays the same: same three members, same pin at A, same load at C. The
only change is that B no longer pushes back, so $B_y = 0$. If the truss can still stand,
the solver will find the forces; if it cannot, it will tell us.

In [ ]:
no_roller = Solver()

T_AB, T_AC, T_BC = Reals('T_AB T_AC T_BC')
A_x, A_y, B_y = Reals('A_x A_y B_y')

# --- JOINT A (0, 0) ---
no_roller.add(A_x + T_AB == 0)
no_roller.add(A_y + T_AC == 0)

# --- JOINT B (3, 0) ---
no_roller.add(-T_AB + T_BC * (-3.0/5.0) == 0)
no_roller.add(B_y + T_BC * (4.0/5.0) == 0)

# --- JOINT C (0, 4) ---
no_roller.add(100 + T_BC * (3.0/5.0) == 0)
no_roller.add(-50 - T_AC + T_BC * (-4.0/5.0) == 0)

# the roller is gone, so B provides no reaction at all
no_roller.add(B_y == 0)

print( no_roller.check() )

"unsat" means there is no set of member forces that keeps every joint in equilibrium
once the roller is gone. Not that we failed to find one — that none exists.

We can follow the solver's reasoning back through the equations. With $B_y = 0$, the
vertical equation at joint B forces $T_{BC} = 0$, and with the diagonal carrying nothing,
the horizontal equation at joint C reduces to $100 = 0$. There is nothing left in the
structure to push back against the sideways load.

The underlying reason is that a pin gives two reactions and a planar structure needs
three to be held still. The roller supplies the third. This is the sort of thing that is
easy to overlook when the numbers happen to work out on a first pass, and the solver
refuses the whole system rather than handing back a partial answer.